# ⚙️ LIGO Engineering Notes

---

## 📑 Table of Contents

- [Loggers](#loggers)
- [GPS Time](#gps-time)
- [MiB vs. MB](#mib-vs-mb)
- [Spark](#spark)

---

## Loggers

Python's `logging` library provides **levels** that control how much gets logged. Each level has a string name, a numeric value, and a constant.

| Level | Numeric Value | Recommended For |
|-------|--------------|-----------------|
| DEBUG | 10 | Development — logs everything |
| INFO | 20 | Production — logs normal operations |
| WARNING | 30 | Potential issues |
| ERROR | 40 | Errors that need attention |
| CRITICAL | 50 | Catastrophic events only |

You can define the level in three equivalent ways:

```python
level="DEBUG"        # string
level=10             # numeric value
level=logging.DEBUG  # constant
```

> **CRITICAL** has the highest numeric value — it is the strictest filter, silencing almost all output and only letting catastrophic events through.

### Why `stream=sys.stdout`?

By default, Python sends all logs to **Standard Error** (`sys.stderr`). Setting `stream=sys.stdout` redirects logs to **Standard Output** instead.

This matters because external systems (like log collectors, CI pipelines, or containers) often treat `stderr` as an error signal. Using `sys.stdout` forces logs down the "normal data" pipe so external systems don't misinterpret them as failures.

```python
logging.basicConfig(level=logging.DEBUG, stream=sys.stdout)
```

---

## GPS Time

> **GPS time** is a continuous integer count of seconds since **January 6, 1980 00:00:00 UTC**.

- It has **no leap seconds** — it never pauses or resets
- It increases monotonically, making it ideal for scientific timestamps
- LIGO uses GPS time to precisely mark every sample in the strain data

---

## MiB vs. MB

Two different standards — both correct, but for different contexts:

| Unit | Base | 1 unit = | Used in |
|------|------|----------|---------|
| MB (Megabyte) | Powers of 10 | 1,000,000 bytes | Marketing, storage, internet speeds |
| MiB (Mebibyte) | Powers of 2 | 2²⁰ = 1,048,576 bytes | OS, memory, low-level computing |

**Why LIGO uses MiB:** computers are binary, and memory naturally aligns with powers of 2 — 2¹⁰ = 1024, 2²⁰, 2³⁰, etc.

### LIGO Example — 4096 sec file at 4096 Hz

- Each sample is **float64** = 64 bits = **8 bytes**
- Total samples: `4096 sec × 4096 Hz = 16,777,216 samples`

```
16,777,216 × 8 bytes = 134,217,728 bytes

÷ 1,000,000     = 134 MB
÷ 1,048,576     = 128 MiB
```

Both are correct — they just use different divisors.

---

## Spark

### Data Size in LIGO

For a **4096 sec file at 4096 Hz**, as shown above, the raw strain data is **134 MB / 128 MiB**.

When storing this in Spark, there are two design choices:

| Option | Description | Recommended? |
|--------|-------------|:---:|
| One giant row | Store entire strain as 1 row with metadata | ❌ No |
| Windowed rows | Split into 2-second windows, one row per window | ✅ Yes |

---

### Why Is One 134 MB Row Bad in Spark?

Spark's parallelism comes from **partitions**:

```
1 partition  →  1 task  →  1 CPU core
```

Executors run **many tasks in parallel** across many cores. So parallelism depends entirely on having **many partitions**.

If the entire signal is stored as **one giant row**:
- The DataFrame has essentially one logical object
- A UDF sees **one row → one function call → one task → one core**
- The rest of your cluster sits idle

> **Spark likes many small/medium records. It dislikes one giant record.**

Even if your cluster has 100 cores, only 1 core does the real work on that signal.

**The fix — windowing:**

Split the signal into 2-second windows before storing. Each window becomes its own row, and Spark can process all windows in parallel across the cluster.

```
4096 sec ÷ 2 sec/window = 2048 rows  →  2048 tasks  →  full parallelism
```

---

### Ideal Partition Size

| Data Type | Recommended Size per Partition |
|-----------|-------------------------------|
| General objects | 100 MB to 256 MB |
| Parquet files | 128 MB to 1 GB |

**LIGO — 2-second window at 4096 Hz:**

```
8192 samples × 8 bytes (float64) = 65,536 bytes ≈ 64 KiB per row
```

64 KiB per row is well within the reasonable range — small enough for Spark to handle many in parallel, large enough to avoid excessive overhead per task.

---

### Spark Task

> A **task** is the smallest unit of work in Spark: one operation applied to one partition.

```
Task = Partition + Operation
```

One task runs on one CPU core. The more partitions you have, the more tasks Spark can distribute across your cluster — and the faster your job runs.

## Bronze Layer — Summary & Key Concepts

### Architecture decision
- Raw scientific arrays (`data.value`) stay **outside Spark** — in NumPy (`.npy`) or GWpy-native (`.hdf5`).
- Spark manages **metadata only**: indices, GPS times, lineage, quality flags.
- Root cause of earlier Spark slowness: millions of Python floats / nested lists / array<double> columns → Python↔JVM serialization overhead. Not the 128 MiB size itself.

### Bronze schema (per window)
| Column | Meaning |
|---|---|
| `window_id` | Sequential window index |
| `start_idx` / `end_idx` | Exact integer slice into raw array (`raw[start_idx:end_idx]`) |
| `gps_start` / `gps_end` | Float GPS time boundaries |
| `window_duration` | Seconds per window (constant) |
| `num_samples` | Samples per window |
| `source` | Data origin (e.g. `GWOSC`) |
| `file_gps_start` / `file_duration` | Identifies which raw file this window belongs to |
| `detector` | H1 / L1 / etc. |

**Why store both `start_idx/end_idx` AND `gps_start/gps_end`:**
`start_idx/end_idx` = integer arithmetic → exact.
`gps_start/gps_end` = float arithmetic → human-readable but rounding-prone.
Storing both avoids re-deriving an integer index from float math downstream.

### Reconstructing the signal
Bronze rows are **pointers**, not data:
```python
raw = np.load("../data/raw/H1_1126259462_4096.npy")
window_signal = raw[start_idx:end_idx]
```
Caching raw data locally (`.npy`) avoids re-fetching from GWOSC on every window (2048x redundant network calls otherwise).

### Spark: generating rows natively (no Python loop)
```python
windows_df = spark.range(window_numbers) \
    .withColumnRenamed("id", "window_id") \
    .withColumn("start_idx", col("window_id") * window_size) \
    .withColumn("end_idx", col("start_idx") + window_size) \
    .withColumn("gps_start", lit(t0) + col("window_id") * window_seconds) \
    .withColumn("window_duration", lit(window_seconds)) \
    .withColumn("gps_end", col("gps_start") + col("window_duration")) \
    .withColumn("num_samples", col("end_idx") - col("start_idx")) \
    .withColumn("source", lit(source)) \
    .withColumn("file_gps_start", lit(t0)) \
    .withColumn("file_duration", lit(len(data.value) // sample_rate)) \
    .withColumn("detector", lit(detector))
```

**Key Spark concepts learned:**
- `spark.range(n)` → generates a distributed, partitioned DataFrame of integers natively in the JVM. No driver-side Python loop, scales to hundreds of millions of rows.
- `withColumn(name, expr)` → `expr` **must** be a `Column` object, not a raw Python value.
- `col("x")` → references an existing column as an expression (runs per-partition, in parallel, in the JVM).
- `lit(x)` → wraps a plain Python/scalar value into a constant `Column` so it can be broadcast to every row.
- Rule: bare Python variables only work as the **first** arg (column name string). Every value arg needs `lit()`; every column reference needs `col()`.
- A Python UDF / `.rdd.map(lambda ...)` still ships rows through Python — defeats the purpose. Native `withColumn` expressions never leave the JVM.

### Writing / reading Parquet
```python
windows_df.write.mode("overwrite").parquet("../data/bronze/h1_windows")

# Always read the FOLDER, never a specific part-file
parquet_df = spark.read.parquet("../data/bronze/h1_windows")
```
- Spark writes a **folder** of part-files, not a single file — part-file names/hashes change between runs.
- `mode("overwrite")` replaces the folder; `mode("append")` for accumulating multiple detectors/segments.
- For multi-detector datasets, consider `partitionBy("detector")` to get `detector=H1/`, `detector=L1/` subfolders in one logical table.

### Gotchas caught
- `window_duration` must be `lit(window_seconds)` (constant), not derived from `gps_start` (would silently compute `gps_end` instead).
- Slicing raw array for a window must use `samples_per_window` (8192), not `sample_rate` (4096) — easy variable mix-up.
- After Parquet round-trip, all columns become `nullable = true` (Parquet doesn't preserve Spark's `nullable=false` constraint) — worth remembering for downstream schema validation.

### Validation checklist
```python
windows_df.printSchema()
windows_df.count()           # expect 2048
row0 = parquet_df.filter(col("window_id") == 0).collect()[0]
assert len(raw[row0.start_idx:row0.end_idx]) == row0.num_samples
```

### Milestone status
✅ Raw signal cached locally (`.npy`)
✅ Bronze DataFrame generated natively via `spark.range`
✅ Bronze written to Parquet and validated via round-trip read
➡️ Next: Silver layer — slice windows from cached raw array, run quality checks (NaN, flat signal, outliers), store quality metadata (not arrays).